In [1]:
import pandas as pd
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import json

In [6]:
%pip install -q dotenv openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
def sample_csv_for_llm(csv_path: str, sample_size: int = 500):
    
    df = pd.read_csv(csv_path)
    
    if len(df) < sample_size:
        sample_size = len(df)
    
    sample_df = df.sample(n=sample_size, random_state=42)
    sample_json = sample_df.to_json(orient='records', date_format='iso')        
    
    return sample_json

In [4]:
file_path = 'datasets/akshaydattatraykhare/diabetes-dataset/versions/1/diabetes.csv'

dataset_title = os.path.splitext(os.path.basename(file_path))[0]

sample_json = sample_csv_for_llm(file_path, 500)

In [5]:
#setup OpenAI API
load_dotenv()
client = OpenAI()

In [6]:
#OpenAI call
def call_openai_api(prompt, model):
    if model == 'o1-mini':
        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]
    else:
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": prompt
            }
        ]

    completion = client.chat.completions.create(
    model=model,
    messages=messages
)
    return completion

In [7]:
RESPONSE_QUERIES = """ 
{
  "queries": [
    {
      "query": "Select data ..."
    },
    {
      "query": "Find datasets ..."
    },
    {
      "query": "Show me data ..."
    }
  ]
}
"""

In [15]:
prompt_to_instructions = f"""
You are the dataset owner of {dataset_title}. You need to provide instructions to the users on how to discover effectively the dataset in a DataSpace platform.
Here you can find a Dataset sample: {sample_json} 
The final users don't have access to the dataset content. So, provide instructions (queries) that they could use to find this dataset.

One example of a query could be: "Find the dataset with entries about cannabis strains and their effects"

Generate as many queries as required to cover the entire dataset content and structure.
Reason step by step:
1. First understand the dataset content and structure.
2. Formulate general queries to show the dataset content.
3. Formulate precise queries to highlight specific findings or limitations.

The output must be a valid JSON object that can be directly loaded by json.loads. It should be a list of queries. An example response is: {RESPONSE_QUERIES}
Provide only the JSON object without additional queries
"""


In [ ]:
list_Q = call_openai_api(prompt_to_instructions, "o1-mini").choices[0].message.content

In [10]:
list_Q = list_Q.replace("```json\n", "").replace("\n```", "")

queries_dict = json.loads(list_Q)

In [11]:
for i, query in enumerate(queries_dict['queries'], start=1):
    print(f"{i}. {query['query']}\n")

1. Find the dataset containing medical records related to diabetes diagnosis and patient metrics such as glucose levels, BMI, age, and outcomes.

2. Search for a diabetes dataset that includes variables like pregnancies, blood pressure, insulin levels, skin thickness, BMI, and diabetes pedigree function.

3. Locate the dataset with entries on diabetic outcomes based on factors including glucose concentration, age, BMI, and insulin measurements.

4. Find a comprehensive diabetes dataset with patient information including age, BMI, glucose levels, blood pressure, and diabetes outcomes.

5. Search for the diabetes dataset used for predicting diabetic outcomes based on variables such as pregnancies, glucose, age, and insulin levels.

6. Retrieve the dataset that has data on pregnancies, glucose, blood pressure, skin thickness, insulin, BMI, diabetes pedigree function, age, and outcome.

7. Find the dataset with diabetes-related data that includes insulin measurements and diabetes pedigree 

In [16]:
list_Q_4omini = call_openai_api(prompt_to_instructions, "gpt-4o-mini").choices[0].message.content

In [17]:
list_Q_4omini

'```json\n{\n  "queries": [\n    {\n      "query": "Find the dataset containing entries about diabetes-related health metrics and patient demographics."\n    },\n    {\n      "query": "Show me datasets related to pregnancy, glucose levels, and blood pressure."\n    },\n    {\n      "query": "Retrieve entries where the BMI is greater than 30."\n    },\n    {\n      "query": "Find the dataset that includes information about insulin levels and diabetes outcomes."\n    },\n    {\n      "query": "Display all entries with age above 50 and their corresponding diabetes outcome."\n    },\n    {\n      "query": "Select data where skin thickness is greater than 30 and glucose levels exceed 150."\n    },\n    {\n      "query": "Find datasets related to the effect of obesity, indicated by BMI, on diabetes outcomes."\n    },\n    {\n      "query": "Show me entries for patients who had more than 5 pregnancies."\n    },\n    {\n      "query": "Retrieve all data with glucose levels below 100."\n    },\

In [18]:
list_Q_4omini = list_Q_4omini.replace("```json\n", "").replace("\n```", "")

queries_dict_4omini = json.loads(list_Q_4omini)

In [19]:
for i, query in enumerate(queries_dict_4omini['queries'], start=1):
    print(f"{i}. {query['query']}\n")

1. Find the dataset containing entries about diabetes-related health metrics and patient demographics.

2. Show me datasets related to pregnancy, glucose levels, and blood pressure.

3. Retrieve entries where the BMI is greater than 30.

4. Find the dataset that includes information about insulin levels and diabetes outcomes.

5. Display all entries with age above 50 and their corresponding diabetes outcome.

6. Select data where skin thickness is greater than 30 and glucose levels exceed 150.

7. Find datasets related to the effect of obesity, indicated by BMI, on diabetes outcomes.

8. Show me entries for patients who had more than 5 pregnancies.

9. Retrieve all data with glucose levels below 100.

10. Find the dataset filtering for individuals with a diabetes pedigree function score above 0.5.

11. Show me data entries of patients aged 30 and younger with a diabetes outcome of 1.

12. Select all records where blood pressure is above 85.

13. Find datasets providing correlations bet